In [ ]:
# ════════════════════════════════════════════
# CELL 1: SETUP & INSTALL
# ════════════════════════════════════════════
import os

os.makedirs("/content/ai-service", exist_ok=True)
print("✅ Folder siap")

!pip install -q -U google-genai fastapi uvicorn pyngrok nest-asyncio requests pydantic
print("✅ Library terinstall")

✅ Folder siap
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 822.5/822.5 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.5/117.5 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.4/71.4 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.3/472.3 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.1/246.1 kB 18.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth 2.53.0 which is inc

In [ ]:
# ════════════════════════════════════════════
# CELL 2: API KEYS
# Isi kedua key di bawah ini
# ════════════════════════════════════════════

GEMINI_API_KEY  = "YOUR_API_KEY"   # https://aistudio.google.com/app/apikey

# Simpan ke environment
os.environ["GEMINI_API_KEY"]   = GEMINI_API_KEY

# Validasi
ok = True
if "ISI_" in GEMINI_API_KEY:
    print("❌ GEMINI_API_KEY belum diisi!")
    ok = False
if ok:
    print("✅ Key tersimpan")

✅ Key tersimpan


In [ ]:
# ════════════════════════════════════════════
# CELL 3: TEST GEMINI
# Pastikan API key valid sebelum lanjut
# ════════════════════════════════════════════
from google import genai

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

try:
    resp = client.models.generate_content(
        model="gemini-2.5-flash",
        contents="Balas hanya dengan: GEMINI OK"
    )
    print(f"✅ Gemini terhubung → {resp.text.strip()}")
except Exception as e:
    print(f"❌ Gagal: {e}")

✅ Gemini terhubung → GEMINI OK


In [ ]:
# ════════════════════════════════════════════
# CELL 4: TULIS interpreter.py
# Tugasnya: ubah angka mentah → label bermakna
# Threshold berdasarkan statistik dataset (n=5100)
# ════════════════════════════════════════════

interpreter_code = '''
def interpret_data_siswa(raw: dict) -> dict:
    hasil = raw.copy()

    # 1. Total jam belajar (study + self_study + online)
    total_belajar = (
        raw["study_hours"] +
        raw["self_study_hours"] +
        raw["online_classes_hours"]
    )
    hasil["total_jam_belajar"] = round(total_belajar, 2)

    if total_belajar >= 10:
        hasil["label_belajar"] = "sangat tinggi"
    elif total_belajar >= 7:
        hasil["label_belajar"] = "cukup baik"
    elif total_belajar >= 4:
        hasil["label_belajar"] = "rendah"
    else:
        hasil["label_belajar"] = "sangat rendah dan mengkhawatirkan"

    # 2. Distraksi digital (sosmed + gaming)
    total_distraksi = raw["social_media_hours"] + raw["gaming_hours"]
    hasil["total_jam_distraksi"] = round(total_distraksi, 2)

    if total_distraksi >= 7:
        hasil["label_distraksi"] = "sangat tinggi"
    elif total_distraksi >= 5:
        hasil["label_distraksi"] = "tinggi"
    elif total_distraksi >= 3:
        hasil["label_distraksi"] = "sedang"
    else:
        hasil["label_distraksi"] = "rendah dan terkontrol"

    # 3. Kualitas tidur
    if raw["sleep_hours"] >= 8:
        hasil["label_tidur"] = "ideal"
    elif raw["sleep_hours"] >= 6:
        hasil["label_tidur"] = "cukup"
    else:
        hasil["label_tidur"] = "kurang dan berisiko"

    # 4. Kesehatan mental (range 0-11, mean=4.97)
    if raw["mental_health_score"] >= 8:
        hasil["label_mental"] = "baik"
    elif raw["mental_health_score"] >= 4:
        hasil["label_mental"] = "cukup"
    else:
        hasil["label_mental"] = "memerlukan perhatian serius"

    # 5. Burnout (range 0-77, mean=13.28)
    if raw["burnout_level"] >= 40:
        hasil["label_burnout"] = "tinggi"
    elif raw["burnout_level"] >= 15:
        hasil["label_burnout"] = "sedang"
    else:
        hasil["label_burnout"] = "rendah"

    # 6. Focus index (range 0-60, mean=14.53)
    if raw["focus_index"] >= 30:
        hasil["label_fokus"] = "tinggi"
    elif raw["focus_index"] >= 10:
        hasil["label_fokus"] = "sedang"
    else:
        hasil["label_fokus"] = "rendah"

    # 7. Performa & risiko dari exam_score (range 0-100, mean=36)
    score = raw["exam_score"]
    if score >= 65:
        hasil["label_performa"] = "baik"
        hasil["label_risiko"]   = "aman"
    elif score >= 35:
        hasil["label_performa"] = "cukup"
        hasil["label_risiko"]   = "perlu perhatian"
    else:
        hasil["label_performa"] = "memerlukan intervensi segera"
        hasil["label_risiko"]   = "berisiko"

    return hasil
'''

with open("/content/ai-service/interpreter.py", "w") as f:
    f.write(interpreter_code)
print("✅ interpreter.py ditulis")

✅ interpreter.py ditulis


In [ ]:
# ════════════════════════════════════════════
# CELL 5: TULIS prompt_engine.py
# 3 fungsi prompt: guru, admin, ortu
# ════════════════════════════════════════════

prompt_code = '''
from interpreter import interpret_data_siswa


def buat_prompt_guru(data_mentah: dict) -> str:
    """Narasi analisis untuk Wali Kelas."""
    d = interpret_data_siswa(data_mentah)
    nama = d.get("nama", "Siswa")
    return f"""Kamu adalah analis akademik profesional untuk wali kelas sekolah menengah di Indonesia.

DATA SISWA:
- Nama             : {nama}, usia {d["age"]} tahun
- Total belajar    : {d["total_jam_belajar"]} jam/hari ({d["label_belajar"]})
- Distraksi digital: {d["total_jam_distraksi"]} jam/hari sosmed+gaming ({d["label_distraksi"]})
- Kualitas tidur   : {d["sleep_hours"]} jam/malam ({d["label_tidur"]})
- Kesehatan mental : skor {d["mental_health_score"]} dari 11 ({d["label_mental"]})
- Burnout          : {d["burnout_level"]} ({d["label_burnout"]})
- Fokus belajar    : {d["focus_index"]} ({d["label_fokus"]})
- Prediksi nilai   : {d["exam_score"]} — {d["label_performa"]}
- Status risiko    : {d["label_risiko"]}

INSTRUKSI:
Tulis analisis untuk Wali Kelas dalam 3 kalimat Bahasa Indonesia formal.
Kalimat 1: Jelaskan kondisi kebiasaan dan gaya hidup siswa berdasarkan data.
Kalimat 2: Jelaskan bagaimana kondisi tersebut berdampak pada performa akademiknya.
Kalimat 3: Berikan 1 rekomendasi tindakan konkret yang bisa dilakukan wali kelas minggu ini.
Jangan gunakan bullet point dan markdown. Tulis sebagai paragraf mengalir."""


def buat_prompt_admin(data_kelas: dict) -> str:
    """Laporan eksekutif ringkas untuk Admin / Kepala Sekolah."""
    return f"""Kamu adalah sistem laporan eksekutif akademik untuk Kepala Sekolah di Indonesia.

RINGKASAN KELAS {data_kelas["nama_kelas"]}:
- Total siswa        : {data_kelas["total_siswa"]} orang
- Siswa aman         : {data_kelas["siswa_aman"]} orang
- Siswa berisiko     : {data_kelas["siswa_berisiko"]} orang
- Rata-rata belajar  : {data_kelas["avg_study_hours"]} jam/hari
- Rata-rata distraksi: {data_kelas["avg_distraksi"]} jam/hari
- Rata-rata burnout  : {data_kelas["avg_burnout"]}
- Rata-rata nilai    : {data_kelas["avg_exam_score"]}
- Faktor risiko utama: {data_kelas["faktor_dominan"]}

INSTRUKSI:
Tulis laporan eksekutif 3 kalimat dalam Bahasa Indonesia formal untuk Kepala Sekolah.
Kalimat 1: Sampaikan kondisi umum kelas berdasarkan data jumlah siswa aman dan berisiko.
Kalimat 2: Jelaskan faktor risiko utama yang perlu mendapat perhatian.
Kalimat 3: Berikan 1 rekomendasi kebijakan strategis yang bisa segera ditindaklanjuti.
Jangan gunakan bullet point dan markdown. Nada: profesional, berbasis data, ringkas."""


def buat_prompt_ortu(data_mentah: dict) -> str:
    """Pesan informatif hangat untuk Orang Tua."""
    d = interpret_data_siswa(data_mentah)
    nama = d.get("nama", "putra/putri Anda")
    return f"""Kamu adalah asisten informasi akademik yang berbicara langsung kepada orang tua siswa.

DATA ANAK:
- Nama             : {nama}
- Kebiasaan belajar: {d["total_jam_belajar"]} jam/hari ({d["label_belajar"]})
- Penggunaan gadget: {d["total_jam_distraksi"]} jam/hari ({d["label_distraksi"]})
- Kualitas tidur   : {d["sleep_hours"]} jam/malam ({d["label_tidur"]})
- Kondisi mental   : {d["label_mental"]}
- Kelelahan        : {d["label_burnout"]}
- Performa akademik: {d["label_performa"]}
- Status           : {d["label_risiko"]}

INSTRUKSI:
Tulis pesan 3 kalimat dalam Bahasa Indonesia yang hangat untuk orang tua.
Kalimat 1: Sampaikan kondisi anak secara jujur namun tidak menakut-nakuti.
Kalimat 2: Sebutkan 1 hal utama yang perlu diperhatikan dan dibantu di rumah.
Kalimat 3: Berikan 1 saran konkret yang mudah dilakukan orang tua sehari-hari.
Hindari angka dan istilah teknis. Jangan gunakan bullet point dan markdown.
Nada: hangat, suportif, tidak menghakimi."""
'''

with open("/content/ai-service/prompt_engine.py", "w") as f:
    f.write(prompt_code)
print("✅ prompt_engine.py ditulis")

✅ prompt_engine.py ditulis


In [ ]:
main_code = '''
import os

from fastapi import APIRouter, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional
from google import genai

from interpreter import interpret_data_siswa
from prompt_engine import buat_prompt_guru, buat_prompt_admin, buat_prompt_ortu

_api_key = os.environ.get("GEMINI_API_KEY", "")
if not _api_key:
    raise RuntimeError("GEMINI_API_KEY tidak ditemukan!")
client = genai.Client(api_key=_api_key)

router = APIRouter()


class DataSiswa(BaseModel):
    nama: Optional[str] = "Siswa"
    age: float
    study_hours: float
    self_study_hours: float
    online_classes_hours: float
    social_media_hours: float
    gaming_hours: float
    sleep_hours: float
    screen_time_hours: float
    mental_health_score: float
    focus_index: float
    burnout_level: float
    productivity_score: float
    exam_score: float


class DataKelas(BaseModel):
    nama_kelas: str
    total_siswa: int
    siswa_aman: int
    siswa_berisiko: int
    avg_study_hours: float
    avg_distraksi: float
    avg_burnout: float
    avg_exam_score: float
    faktor_dominan: str


def generate(prompt: str) -> str:
    try:
        resp = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt
        )
        return resp.text.strip()
    except Exception as e:
        raise HTTPException(status_code=503, detail=f"Gemini error: {str(e)}")


@router.get("/health", tags=["Health"])
def health():
    try:
        r = client.models.generate_content(
            model="gemini-2.5-flash", contents="Output harus persis: OK"
        )
        return {"status": "ok", "gemini": "terhubung", "ping": r.text.strip()}
    except Exception as e:
        return {"status": "error", "gemini": "tidak terhubung", "detail": str(e)}


@router.post("/narasi/guru", tags=["Narasi"])
def narasi_guru(data: DataSiswa):
    return {
        "status": "success",
        "persona": "guru",
        "narasi": generate(buat_prompt_guru(data.dict()))
    }


@router.post("/narasi/admin", tags=["Narasi"])
def narasi_admin(data: DataKelas):
    return {
        "status": "success",
        "persona": "admin",
        "narasi": generate(buat_prompt_admin(data.dict()))
    }


@router.post("/narasi/ortu", tags=["Narasi"])
def narasi_ortu(data: DataSiswa):
    return {
        "status": "success",
        "persona": "ortu",
        "narasi": generate(buat_prompt_ortu(data.dict()))
    }
'''

with open("/content/ai-service/narasi.py", "w") as f:
    f.write(main_code.strip())
print("✅ narasi.py ditulis")


✅ narasi.py ditulis


In [ ]:
requirements = """
fastapi
uvicorn
google-genai
pydantic
python-dotenv
"""

with open("/content/ai-service/requirements.txt","w") as f:
    f.write(requirements)

print("✅ requirements.txt dibuat")

✅ requirements.txt dibuat


In [ ]:
readme = """
# SNBPredict AI Service

AI Narrative Service menggunakan FastAPI dan Gemini.

Endpoint:

POST /narasi/guru

POST /narasi/admin

POST /narasi/ortu

Environment Variable:

GEMINI_API_KEY
"""

with open("/content/ai-service/README.md","w") as f:
    f.write(readme)

print("✅ README.md dibuat")

✅ README.md dibuat


In [ ]:
import shutil

shutil.make_archive(
    "/content/SNBPredict_AI_Service",
    "zip",
    "/content/ai-service"
)

print("✅ ZIP berhasil dibuat")

✅ ZIP berhasil dibuat


In [ ]:
from google.colab import files

files.download(
    "/content/SNBPredict_AI_Service.zip"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>